In [1]:
import os
os.chdir('../../../../..')

In [2]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
from kmedoids import KMedoids

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [ ]:
qm9 = QM9Dataset(limit=5000, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"], add_acsf=True)
df = qm9.load()
molecules = qm9.get_molecules()

2026-04-15 15:48:55.359 | INFO     | src.datasets:load:493 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-15 15:48:55.707 | INFO     | src.datasets:_sample_qm9_df:685 - QM9 sampling complete: strategy=stratified, requested_limit=5500, returned_rows=5500.
2026-04-15 15:48:55.708 | INFO     | src.datasets:_add_requested_descriptors:126 - Applying requested QM9 descriptors to sampled dataframe (rows=5500).
2026-04-15 15:48:55.708 | INFO     | src.features:compute_acsf:193 - Computing ACSF (rcut=6.0)...
2026-04-15 15:49:36.426 | SUCCESS  | src.datasets:add_acsf:850 - Added ACSF embeddings.
2026-04-15 15:49:36.427 | INFO     | src.datasets:_add_requested_descriptors:149 - Added descriptor column(s): ['acsf_embedding']
2026-04-15 15:49:36.439 | INFO     | src.datasets:_drop_rows_with_null_required_descriptors:578 - Dropped QM9 rows with null/empty descriptor vectors: dropped=23, remaining=5477, descriptor_cols=['acsf_embedding'].
2026-04-15 15:49:36.440 | IN

In [7]:
len(molecules[0:2])

2

In [8]:
plot_molecules_with_py3dmol(molecules[0:3])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
dist_matrix = qm9.get_distance_matrix(
    descriptor="acsf",
    dist_type="euclidean",
)

2026-04-15 15:50:59.442 | INFO     | src.distance:_compute_and_save:34 - Computing euclidean distance matrix...
2026-04-15 15:50:59.898 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_acsf_euclidean.npy


# Determining the best number of clusters for each clustering method

In [11]:
out = evaluate_distance_matrix_clustering_sweep(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    dataset_name="qm9",
)

Evaluation k:   0%|          | 0/19 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [12]:
# find the n molecules that are not on the diagonal with the smallest distance
n = 10
# Get the indices of the upper triangle (excluding diagonal)
triu_indices = np.triu_indices_from(dist_matrix, k=1)
# Get the distances and corresponding molecule pairs
distances = dist_matrix[triu_indices]
molecule_pairs = list(zip(triu_indices[0], triu_indices[1]))
# Get the indices of the n smallest distances
smallest_indices = np.argsort(distances)[:n]
# Get the corresponding molecule pairs for the n smallest distances
closest_pairs = [molecule_pairs[i] for i in smallest_indices]
print("Closest molecule pairs (indices):", closest_pairs)
mols = [(molecules[idx1], molecules[idx2]) for idx1, idx2 in closest_pairs]

Closest molecule pairs (indices): [(np.int64(1332), np.int64(1333)), (np.int64(1429), np.int64(1430)), (np.int64(3300), np.int64(3313)), (np.int64(4398), np.int64(4418)), (np.int64(2814), np.int64(2845)), (np.int64(3389), np.int64(3698)), (np.int64(3621), np.int64(3622)), (np.int64(3674), np.int64(3675)), (np.int64(4183), np.int64(4188)), (np.int64(1473), np.int64(1488))]


In [13]:
print(mols[0])

(Atoms(symbols='H3CH3NHOCNCNC2', pbc=False, initial_charges=..., mass=..., partial_charge=...), Atoms(symbols='H3CH3NHOCNCNC2', pbc=False, initial_charges=..., mass=..., partial_charge=...))


In [14]:
plot_molecules_with_py3dmol(mols[1])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Hiercical Clustering on Distance Matrix

In [16]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=3, linkage='complete')
labels_hier = model_hier.fit_predict(dist_matrix)
df = df.with_columns(labels_hier=labels_hier)

In [17]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [18]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_hier,
    clustering_method="hierarchical"
)

2026-04-15 15:52:10.435 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_hierarchical_projection.png


{'coords': array([[-42.87330632,  26.97616543],
        [ 72.57577087,   0.38978408],
        [ 99.66756854,  32.53057816],
        ...,
        [-33.04061161,  41.09393408],
        [-45.10445665,  39.92296242],
        [-31.10718847, -25.59146542]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_hierarchical_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'hierarchical'}

In [19]:
average_numeric_by_cluster(df, "labels_hier")

labels_hier,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,814,1.851738,14.45086,118.642506,-0.103194,54.243243,0.733513,12.893951,8.566339,1.109337,0.504914,2.009945,1.605651,0.140798,0.525233,0.33397,1.103194,2.756757,5.318182,0.824324,2.595823,1.765356,6.307125,26.886978,1.262112,0.003686,0.162162,0.108108,0.308354,0.257985,0.001229,0.052826,0.148649,0.313268,0.001229,3.380835,3.316457,69.437469,-6.489795,-0.682217,5.807645,1125.029184,2.966725,-11361.508652,-11361.292193,-11361.266499,-11362.407162,28.652026,-64.16806,-64.508595,-64.85427,-59.928457,4.195144,1.457211,1.095235,36.117936,47.542998,16.339066
1,3150,2.12,18.238413,123.248889,-0.059048,36.959683,0.892231,12.838172,8.785397,1.66381,0.050794,2.063903,2.487619,0.065686,0.161066,0.773247,0.973333,2.036825,6.48381,0.444444,1.007302,4.906667,6.374286,38.710159,1.262036,0.000952,0.419365,0.010159,0.128889,0.130159,0.002222,0.048889,0.14127,0.654921,0.0,2.426984,2.732399,74.830917,-6.562076,0.346257,6.908291,1198.457931,4.114659,-11243.37496,-11243.13964,-11243.113942,-11244.290286,32.095858,-76.618787,-77.086385,-77.52945,-71.287569,3.313139,1.377942,1.116646,83.650794,5.079365,11.269841
2,1036,2.231232,21.206564,123.611969,0.671815,17.390927,0.93643,12.781541,8.875483,2.15444,0.002896,2.096841,2.358108,0.021435,0.063768,0.914797,0.584942,1.069498,7.655405,0.169884,0.494208,6.930502,6.248069,47.296332,1.262622,0.000965,0.354247,0.0,0.022201,0.007722,0.0,0.008687,0.072394,0.471042,0.0,1.280888,1.798122,81.713378,-6.513106,1.301802,7.814979,1193.819869,5.063416,-10524.784472,-10524.547741,-10524.522067,-10525.689485,33.755863,-87.546984,-88.127608,-88.646995,-81.349363,2.943496,1.436879,1.197572,94.401544,0.289575,5.30888


# KMedoids

In [20]:
model_km = KMedoids(n_clusters=3, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
df = df.with_columns(labels_km=labels_km)

In [21]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [22]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_km,
    clustering_method="kmedoids"
)

2026-04-15 15:52:46.900 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_kmedoids_projection.png


{'coords': array([[-42.87330632,  26.97616543],
        [ 72.57577087,   0.38978408],
        [ 99.66756854,  32.53057816],
        ...,
        [-33.04061161,  41.09393408],
        [-45.10445665,  39.92296242],
        [-31.10718847, -25.59146542]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_kmedoids_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'kmedoids'}

In [23]:
average_numeric_by_cluster(df, "labels_km")

labels_km,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,pct_aliphatic_ring,pct_aromatic,pct_acyclic
u64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1872,2.22956,20.729167,123.803953,0.365919,23.094551,0.90899,12.82397,8.837607,1.929487,0.004274,2.078825,2.668269,0.025234,0.071715,0.903051,0.801816,1.296474,7.438568,0.186966,0.524573,6.495192,6.353098,46.161325,1.258793,0.000534,0.384615,0.0,0.077991,0.043803,0.001068,0.009615,0.074252,0.488782,0.0,1.630876,2.079801,80.260994,-6.492458,1.125438,7.617903,1219.679637,4.90893,-10701.801887,-10701.561225,-10701.53554,-10702.715051,33.770385,-85.3295,-85.887761,-86.394874,-79.278029,3.051123,1.395178,1.154418,1.525107,91.933761,0.42735,7.638889
1,2046,2.096535,17.686217,123.254154,-0.095797,38.506843,0.911055,12.818095,8.798143,1.751222,0.032258,2.074707,2.272727,0.074899,0.163168,0.761933,0.917889,2.183773,6.261486,0.509286,1.024438,4.753666,6.2913,37.059629,1.266241,0.000489,0.44868,0.005865,0.106061,0.120235,0.001955,0.062561,0.173021,0.745357,0.0,2.510753,2.839979,73.483759,-6.633093,0.174439,6.807512,1170.126234,3.936367,-11375.972338,-11375.741353,-11375.715657,-11376.882888,31.52446,-74.900772,-75.351439,-75.780304,-69.722766,3.290948,1.402805,1.139756,1.013685,86.950147,3.225806,9.824047
2,1082,1.879505,14.965804,119.160813,-0.058226,52.288355,0.750556,12.888447,8.592421,1.091497,0.462107,2.008602,1.7939,0.13239,0.492484,0.375125,1.100739,2.655268,5.497227,0.790203,2.513863,2.022181,6.396488,28.266174,1.260312,0.004621,0.168207,0.099815,0.292976,0.277264,0.001848,0.055453,0.136784,0.338262,0.000924,3.266174,3.202887,70.515896,-6.446968,-0.53574,5.911251,1155.63315,3.122426,-11330.465197,-11330.243761,-11330.218061,-11331.370771,29.277793,-65.893579,-66.248964,-66.607879,-61.51044,4.118035,1.417176,1.068974,0.270795,37.615527,43.992606,18.391867


# Spectral

In [ ]:
model_spectral = SpectralClustering(
                n_clusters=6,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)

In [ ]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [ ]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_spectral,
    clustering_method="spectral"
)

2026-04-15 08:37:08.191 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_spectral_projection.png


{'coords': array([[-42.873386 ,  26.976103 ],
        [ 72.57561  ,   0.3898516],
        [ 99.66764  ,  32.530502 ],
        ...,
        [-33.040558 ,  41.093994 ],
        [-45.104256 ,  39.92297  ],
        [-31.107193 , -25.591467 ]], shape=(5000, 2), dtype=float32),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_spectral_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'spectral'}

In [ ]:
average_numeric_by_cluster(df, "labels_spectral")

Hierarchical Clustering
shape: (6, 57)
┌─────────────────┬───────┬───────────┬────────────┬──────────┬───────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬──────────┬───────────┬───────────┬───────────┬──────────┬─────────────┬──────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────┬────────────┬────────────┬────────────┬────────────┬──────────┬──────────┬──────────┬─────────────┬───────────┐
│ labels_spectral ┆ count ┆ num_atoms ┆ mol_weight ┆ logp     ┆ tpsa      ┆ election_affinity ┆ ionization_energies ┆ num_heavy_atoms ┆ 

# DBSCAN 

In [45]:
model_db = DBSCAN(
    eps=0.6,
    min_samples=2,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db, return_counts=True))

(array([-1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10]), array([  30, 4927,    2,   15,    4,    2,    8,    3,    2,    3,    2,
          2]))


In [46]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [47]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_db,
    clustering_method="dbscan"
)

2026-04-15 15:55:51.511 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_dbscan_projection.png


{'coords': array([[-42.87330632,  26.97616543],
        [ 72.57577087,   0.38978408],
        [ 99.66756854,  32.53057816],
        ...,
        [-33.04061161,  41.09393408],
        [-45.10445665,  39.92296242],
        [-31.10718847, -25.59146542]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_dbscan_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'dbscan'}

In [48]:
average_numeric_by_cluster(df, "labels_db")

labels_db,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,labels_km,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-1,30,1.838803,12.366667,111.3,-0.266667,53.666667,0.582404,13.166169,7.9,1.166667,0.533333,2.013343,1.033333,0.131111,0.51627,0.352619,0.933333,3.0,4.3,0.6,1.7,1.733333,5.666667,23.433333,1.269671,0.033333,0.233333,0.066667,0.3,0.233333,0.0,0.033333,0.033333,0.266667,0.0,3.866667,2.6359,59.436334,-7.042942,-0.854165,6.188958,868.940405,2.424126,-11045.315186,-11045.131559,-11045.105762,-11046.165332,24.497267,-54.119524,-54.412372,-54.704468,-50.500578,5.738337,1.897798,1.467094,0.233333,1.7,33.333333,46.666667,20.0
0,4927,2.10453,18.323118,122.721534,0.088695,35.413233,0.879493,12.831917,8.779176,1.685813,0.107571,2.063017,2.333063,0.067503,0.194402,0.738095,0.916582,1.933631,6.57195,0.444895,1.146134,4.866653,6.344226,38.805155,1.262077,0.001218,0.367566,0.022935,0.133956,0.124619,0.001624,0.04039,0.128679,0.565659,0.0,2.321494,2.627326,75.596284,-6.531825,0.399196,6.931019,1188.830828,4.149805,-11112.478022,-11112.244881,-11112.219189,-11113.38915,31.97782,-77.180544,-77.653603,-78.098846,-71.82233,3.348236,1.398277,1.127032,1.057642,0.827481,79.013599,10.33083,10.655571
1,2,1.763636,13.0,108.5,0.0,64.5,0.784607,13.076501,7.5,0.0,0.0,1.842424,4.0,0.0,0.708333,0.291667,0.5,2.5,4.0,0.0,2.5,1.0,7.0,23.0,1.248416,0.0,0.0,0.0,0.0,0.5,0.0,0.5,0.0,1.0,0.0,4.0,2.9738,56.375,-7.111696,-1.178253,5.934803,1247.413269,2.629327,-11069.0,-11068.779297,-11068.753906,-11069.942383,26.154,-55.124903,-55.40526,-55.713657,-51.457008,10.46832,1.11406,1.054135,0.0,2.0,0.0,0.0,100.0
2,15,1.607148,11.466667,107.866667,-0.266667,66.933333,0.170728,13.293639,7.8,1.066667,1.066667,2.011111,1.066667,0.15,0.75,0.1,0.866667,4.2,3.933333,0.533333,2.266667,0.333333,5.8,18.466667,1.262524,0.0,0.0,0.2,0.466667,0.0,0.0,0.0,0.0,0.0,0.0,4.666667,4.292713,58.546,-6.89083,-1.228322,5.662145,829.585537,2.192811,-10592.670898,-10592.497201,-10592.471745,-10593.509375,22.9176,-50.054759,-50.32324,-50.592193,-46.682113,6.097457,1.915449,1.437949,0.0,2.0,0.0,100.0,0.0
3,4,1.807143,13.0,123.5,-0.5,72.25,0.936373,13.001237,8.5,0.5,0.5,1.928571,2.0,0.0,0.729167,0.270833,1.0,4.25,4.5,0.0,2.75,1.0,6.25,23.5,1.259979,0.0,0.0,0.25,0.25,0.75,0.0,0.25,0.25,0.5,0.0,4.75,4.6397,57.32,-6.920536,-1.014985,5.904871,1180.312012,2.473705,-13126.206543,-13125.983643,-13125.95752,-13127.11792,28.079,-57.866344,-58.14431,-58.452682,-54.053913,4.839343,1.35777,1.043265,0.0,2.0,0.0,50.0,50.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6,3,1.42735,12.333333,109.333333,0.0,24.666667,1.118412,12.271584,8.333333,0.0,0.0,1.837607,0.666667,0.761905,0.095238,0.142857,0.666667,1.333333,1.666667,5.333333,0.666667,1.0,7.333333,17.666667,1.246422,0.0,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.333333,0.926167,75.636665,-7.296279,-1.103875,6.192404,1525.466431,2.209211,-10045.867188,-10045.620443,-10045.594401,-10046.813802,31.112667,-58.891087,-59.119888,-59.411123,-55.365351,4.711587,1.162097,0.904023,0.0,2.0,0.0,0.0,100.0
7,2,2.236074,27.5,121.0,3.0,0.0,0.911153,12.875904,8.5,0.0,0.0,1.927056,7.5,0.0,0.0,1.0,0.0,0.0,8.5,0.0,0.0,8.5,7.0,61.5,1.225197,0.0,0.0,0.0,